In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import os

# Paths are relative to the repo root - run this notebook from there.


In [ ]:
# this ensures that the current MacOS version is at least 12.3+
print(torch.backends.mps.is_available())

# this ensures that the current current PyTorch installation was built with MPS activated.
print(torch.backends.mps.is_built())

In [ ]:
# Set datatype and device ("mps" instead of "cuda")
dtype = torch.float
device = torch.device("mps")

In [ ]:
# Specify model to use:
model_name = 'microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract'

### Load in data

In [ ]:
# load in articles df
train_df = pd.read_csv("./scripts/pubmed_output/healthbase_articles_balanced_train_CI.csv")
val_df = pd.read_csv("./scripts/pubmed_output/healthbase_articles_balanced_val_CI.csv")
test_df = pd.read_csv("./scripts/pubmed_output/healthbase_articles_balanced_test_CI.csv")

In [ ]:
train_df.head(3)

In [ ]:
# Check label balance
print(train_df["case"].value_counts())
train_df.groupby('case').size().plot(kind='pie',
                                       y = "v1",
                                       label = "Type",
                                       autopct='%1.1f%%')

In [ ]:
# get arrays of sentences
sentences_train = train_df.CI_TiAb.values
sentences_val = val_df.CI_TiAb.values
sentences_test = test_df.CI_TiAb.values


In [ ]:
# get labels matching the articles
labels_train = train_df.case.to_numpy()
labels_val = val_df.case.to_numpy()
labels_test = test_df.case.to_numpy()

### Check results tokenizer

In [ ]:
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained(model_name)

In [ ]:
# Print the original sentence.
print(' Original: ', sentences_train[0])

# Print the sentence split into tokens.
print('Tokenized: ', tokenizer.tokenize(sentences_train[0]))

# Print the sentence mapped to token ids.
print('Token IDs: ', tokenizer.convert_tokens_to_ids(tokenizer.tokenize(sentences_train[0])))

### Tokenize all sentences
* adapted from https://colab.research.google.com/drive/1pTuQhug6Dhl9XalKB0zUGf4FIdYFlpcX#scrollTo=2bBdb3pt8LuQ

In [ ]:
# set max length BERT
max_length = 512

In [ ]:
type(sentences_train)

In [ ]:
# Function to tokenize and map sentences to their word IDs

def tokenize_and_map(sentences, labels): 

    # Tokenize all of the sentences and map the tokens to their word IDs.
    input_ids = []
    attention_masks = []

    # For every sentence...
    for sent in sentences:
        # `encode_plus` will:
        #   (1) Tokenize the sentence.
        #   (2) Prepend the `[CLS]` token to the start.
        #   (3) Append the `[SEP]` token to the end.
        #   (4) Map tokens to their IDs.
        #   (5) Pad or truncate the sentence to `max_length`
        #   (6) Create attention masks for [PAD] tokens.
        encoded_dict = tokenizer.encode_plus(
                            sent,                      # Sentence to encode.
                            truncation=True,           # add truncation
                            add_special_tokens = True, # Add '[CLS]' and '[SEP]'
                            max_length = max_length,   # Pad & truncate all sentences.
                            pad_to_max_length = True,
                            return_attention_mask = True,   # Construct attn. masks.
                            return_tensors = 'pt',     # Return pytorch tensors.
                    )
        
        # Add the encoded sentence to the list.    
        input_ids.append(encoded_dict['input_ids'])
        
        # And its attention mask (simply differentiates padding from non-padding).
        attention_masks.append(encoded_dict['attention_mask'])

    # Convert the lists into tensors.
    input_ids = torch.cat(input_ids, dim=0)
    attention_masks = torch.cat(attention_masks, dim=0)
    labels = torch.tensor(labels)

    return input_ids, attention_masks, labels

In [ ]:
input_ids_train, attention_masks_train, labels_train = tokenize_and_map(sentences_train, labels_train)
input_ids_val, attention_masks_val, labels_val = tokenize_and_map(sentences_val, labels_val)
input_ids_test, attention_masks_test, labels_test = tokenize_and_map(sentences_test, labels_test)

## Train - val - test as Tensor Datasets

In [ ]:
from torch.utils.data import TensorDataset

# Combine the inputs into a TensorDataset for train/val/test set.
train_dataset = TensorDataset(input_ids_train, attention_masks_train, labels_train)
val_dataset = TensorDataset(input_ids_val, attention_masks_val, labels_val)
test_dataset = TensorDataset(input_ids_test, attention_masks_test, labels_test)

### Create iterator to loop over data
* iterator requires less memory than for loop, because data doesn't need to be fully loaded in memory

In [ ]:
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler

# The DataLoader needs to know our batch size for training, so we specify it 
# here. For fine-tuning BERT on a specific task, the authors recommend a batch 
# size of 16 or 32. 
batch_size = 16

In [ ]:
# Create the DataLoaders for our training and validation sets.
# We'll take training samples in random order. 
train_dataloader = DataLoader(
            train_dataset,  # The training samples.
            sampler = RandomSampler(train_dataset), # Select batches randomly
            batch_size = batch_size # Trains with this batch size.
        )

# For validation the order doesn't matter, so we'll just read them sequentially.
validation_dataloader = DataLoader(
            val_dataset, # The validation samples.
            sampler = SequentialSampler(val_dataset), # Pull out batches sequentially.
            batch_size = batch_size # Evaluate with this batch size.
        )

### Load model and head for classification
* We'll be using BertForSequenceClassification. This is the normal BERT model with an added single linear layer on top for classification that we will use as a sentence classifier. As we feed input data, the entire pre-trained BERT model and the additional untrained classification layer is trained on our specific task.

In [ ]:
from transformers import BertForSequenceClassification, AdamW, BertConfig

# Load BertForSequenceClassification, the pretrained BERT model with a single 
# linear classification layer on top. 
model = BertForSequenceClassification.from_pretrained(
    model_name, 
    num_labels = 2, # The number of output labels--2 for binary classification.
                    # You can increase this for multi-class tasks.   
    output_attentions = False, # Whether the model returns attentions weights.
    output_hidden_states = False, # Whether the model returns all hidden-states.
)


In [ ]:
# Tell pytorch to run this model on the GPU.
device = torch.device('mps')
model.to(device)
device

## Optimizer and learning rate scheduler
#### For the purposes of fine-tuning, the authors recommend choosing from the following values (from Appendix A.3 of the BERT paper):
* Batch size: 16, 32 
* Learning rate (Adam): 5e-5, 3e-5, 2e-5
* Number of epochs: 2, 3, 4
* eps: 1e-8 ( "a very small number to prevent any division by zero in the implementation" )

In [ ]:
# Note: AdamW is a class from the huggingface library (as opposed to pytorch) 
# I believe the 'W' stands for 'Weight Decay fix"
optimizer = AdamW(model.parameters(),
                  lr = 2e-5, 
                  eps = 1e-8 
                )

# Number of training epochs. The BERT authors recommend between 2 and 4. 
epochs = 4


In [ ]:
from transformers import get_linear_schedule_with_warmup

# Total number of training steps is [number of batches] x [number of epochs]. 
# (Note that this is not the same as the number of training samples).
total_steps = len(train_dataloader) * epochs

# Create the learning rate scheduler.
scheduler = get_linear_schedule_with_warmup(optimizer, 
                                            num_warmup_steps = 0, # Default value in run_glue.py
                                            num_training_steps = total_steps)

## Training loop
* PyTorch docs: https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html#sphx-glr-beginner-blitz-cifar10-tutorial-py

In [ ]:
# Function to calculate the accuracy of our predictions vs labels
def flat_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

In [ ]:
# Helper function for formatting elapsed times as hh:mm:ss
import time
import datetime

def format_time(elapsed):
    '''
    Takes a time in seconds and returns a string hh:mm:ss
    '''
    # Round to the nearest second.
    elapsed_rounded = int(round((elapsed)))
    
    # Format as hh:mm:ss
    return str(datetime.timedelta(seconds=elapsed_rounded))


### Train model
* This training code is based on the `run_glue.py` script here:
* https://github.com/huggingface/transformers/blob/5bfcd0485ece086ebcbed2d008813037968a9e58/examples/run_glue.py#L128



In [ ]:
import random
import numpy as np

# Set the seed value all over the place to make this reproducible.
seed_val = 42

random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)

# Store a number of quantities such as training and validation loss, 
# validation accuracy, and timings.
training_stats = []

# Measure the total training time for the whole run.
total_t0 = time.time()

# For each epoch...
for epoch_i in range(0, epochs):
    
    # ========================================
    #               Training
    # ========================================
    
    # Perform one full pass over the training set.

    print("")
    print('======== Epoch {:} / {:} ========'.format(epoch_i + 1, epochs))
    print('Training...')

    # Measure how long the training epoch takes.
    t0 = time.time()

    # Reset the total loss for this epoch.
    total_train_loss = 0

    # Put the model into training mode: `dropout` and `batchnorm` layers behave differently during training
    # vs. test (source: https://stackoverflow.com/questions/51433378/what-does-model-train-do-in-pytorch)
    model.train()

    # For each batch of training data...
    for step, batch in enumerate(train_dataloader):

        # Progress update every 40 batches.
        if step % 40 == 0 and not step == 0:
            # Calculate elapsed time in minutes.
            elapsed = format_time(time.time() - t0)
            
            # Report progress.
            print('  Batch {:>5,}  of  {:>5,}.    Elapsed: {:}.'.format(step, len(train_dataloader), elapsed))

        # Unpack training batch from dataloader. Copy each tensor to the GPU using the `to` method.
        # `batch` contains three pytorch tensors:
        #   [0]: input ids 
        #   [1]: attention masks
        #   [2]: labels 

        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)

        # Always clear any previously calculated gradients before performing a
        # backward pass. PyTorch doesn't do this automatically because 
        # accumulating the gradients is "convenient while training RNNs". 
        # (source: https://stackoverflow.com/questions/48001598/why-do-we-need-to-call-zero-grad-in-pytorch)
        model.zero_grad()        

        # Perform a forward pass (evaluate the model on this training batch).
        # In PyTorch, calling `model` will in turn call the model's `forward` 
        # function and pass down the arguments. The `forward` function is 
        # documented here: 
        # https://huggingface.co/transformers/model_doc/bert.html#bertforsequenceclassification
        # The results are returned in a results object, documented here:
        # https://huggingface.co/transformers/main_classes/output.html#transformers.modeling_outputs.SequenceClassifierOutput
        # Specifically, we'll get the loss (because we provided labels) and the
        # "logits"--the model outputs prior to activation.
        result = model(b_input_ids, 
                       token_type_ids=None, 
                       attention_mask=b_input_mask, 
                       labels=b_labels,
                       return_dict=True)

        loss = result.loss
        logits = result.logits

        # Accumulate the training loss over all of the batches so that we can
        # calculate the average loss at the end. `loss` is a Tensor containing a
        # single value; the `.item()` function just returns the Python value 
        # from the tensor.
        total_train_loss += loss.item()

        # Perform a backward pass to calculate the gradients.
        loss.backward()

        # Clip the norm of the gradients to 1.0. -> This is to help prevent the "exploding gradients" problem.
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        # Update parameters and take a step using the computed gradient.
        # The optimizer dictates the "update rule"--how the parameters are
        # modified based on their gradients, the learning rate, etc.
        optimizer.step()

        # Update the learning rate.
        scheduler.step()

    # Calculate the average loss over all of the batches.
    avg_train_loss = total_train_loss / len(train_dataloader)            
    
    # Measure how long this epoch took.
    training_time = format_time(time.time() - t0)

    print("")
    print("  Average training loss: {0:.2f}".format(avg_train_loss))
    print("  Training epoch took: {:}".format(training_time))
        
    # ========================================
    #               Validation
    # ========================================
    # After the completion of each training epoch, measure our performance on
    # our validation set.

    print("")
    print("Running Validation...")

    t0 = time.time()

    # Put the model in evaluation mode--the dropout layers behave differently
    # during evaluation.
    model.eval()

    # Tracking variables 
    total_eval_accuracy = 0
    total_eval_loss = 0
    nb_eval_steps = 0

    # Evaluate data for one epoch
    for batch in validation_dataloader:
        
        # Unpack this training batch from our dataloader. 
        #
        # As we unpack the batch, we'll also copy each tensor to the GPU using 
        # the `to` method.
        #
        # `batch` contains three pytorch tensors:
        #   [0]: input ids 
        #   [1]: attention masks
        #   [2]: labels 
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)
        
        # Tell pytorch not to bother with constructing the compute graph during
        # the forward pass, since this is only needed for backprop (training).
        with torch.no_grad():        

            # Forward pass, calculate logit predictions.
            # token_type_ids is the same as the "segment ids", which 
            # differentiates sentence 1 and 2 in 2-sentence tasks.
            result = model(b_input_ids, 
                           token_type_ids=None, 
                           attention_mask=b_input_mask,
                           labels=b_labels,
                           return_dict=True)

        # Get the loss and "logits" output by the model. The "logits" are the 
        # output values prior to applying an activation function like the 
        # softmax.
        loss = result.loss
        logits = result.logits
            
        # Accumulate the validation loss.
        total_eval_loss += loss.item()

        # Move logits and labels to CPU
        logits = logits.detach().cpu().numpy()
        label_ids = b_labels.to('cpu').numpy()

        # Calculate the accuracy for this batch of test sentences, and
        # accumulate it over all batches.
        total_eval_accuracy += flat_accuracy(logits, label_ids)
        

    # Report the final accuracy for this validation run.
    avg_val_accuracy = total_eval_accuracy / len(validation_dataloader)
    print("  Accuracy: {0:.2f}".format(avg_val_accuracy))

    # Calculate the average loss over all of the batches.
    avg_val_loss = total_eval_loss / len(validation_dataloader)
    
    # Measure how long the validation run took.
    validation_time = format_time(time.time() - t0)
    
    print("  Validation Loss: {0:.2f}".format(avg_val_loss))
    print("  Validation took: {:}".format(validation_time)) 

    # At the end of each epoch save model
    output_dir = './epochs_model_save/'

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    print("Saving epoch model to %s" % output_dir)

    model_save_path = os.path.join(output_dir, f"model_epoch_{epoch_i + 1}.pt")
    torch.save(model.state_dict(), model_save_path)

    print(f"Model saved to ==> {model_save_path}")
    

    # Record all statistics from this epoch.
    training_stats.append(
        {
            'epoch': epoch_i + 1,
            'Training Loss': avg_train_loss,
            'Valid. Loss': avg_val_loss,
            'Valid. Accur.': avg_val_accuracy,
            'Training Time': training_time,
            'Validation Time': validation_time
        }
    )

print("")
print("Training complete!")

print("Total training took {:} (h:mm:ss)".format(format_time(time.time()-total_t0)))

## Summarize results

In [ ]:
# Create a DataFrame from our training statistics.
df_stats = pd.DataFrame(data=training_stats)

# Use the 'epoch' as the row index.
df_stats = df_stats.set_index('epoch')

# Display the table.
df_stats

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

# Use plot styling from seaborn.
sns.set(style='darkgrid')

# Increase the plot size and font size.
sns.set(font_scale=1.5)
plt.rcParams["figure.figsize"] = (12,6)

# Plot the learning curve.
plt.plot(df_stats['Training Loss'], 'b-o', label="Training")
plt.plot(df_stats['Valid. Loss'], 'g-o', label="Validation")

# Label the plot.
plt.title("Training & Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.xticks([1, 2, 3, 4])

plt.show()

### Load version of model based on best score during epochs

In [ ]:
# Specify the epoch number
epoch_number_to_load = 1

# Define model architecture. This should be the same architecture used during training.
# Load BertForSequenceClassification, the pretrained BERT model with a single linear classification layer on top. 
from transformers import BertForSequenceClassification, AdamW, BertConfig
model = BertForSequenceClassification.from_pretrained(
    model_name, 
    num_labels = 2, # The number of output labels--2 for binary classification.
                    # You can increase this for multi-class tasks.   
    output_attentions = False, # Whether the model returns attentions weights.
    output_hidden_states = False, # Whether the model returns all hidden-states.
)

# Create the path for the model using the same naming convention
model_path = os.path.join('./epochs_model_save/', f'model_epoch_{epoch_number_to_load}.pt')

# Load the model's state dictionary
model_state_dict = torch.load(model_path, map_location=device)  # The map_location ensures that the model is loaded to the right device

# Load the state dictionary into the model
model.load_state_dict(model_state_dict)

# Move the model to the device
model.to(device)
device



In [ ]:
# # Load previously saved model

# # path to the model
# # output_dir = './model_save/'
# output_dir = './model_CI_added_save_batch_16_1epochs'

# # Load model and tokenizer
# model =  BertForSequenceClassification.from_pretrained(output_dir)
# tokenizer = BertTokenizer.from_pretrained(output_dir)

# # Copy the model to the GPU.
# model.to(device)
# device

## Predictions on Test Set
* We'll evaluate predictions using Matthew's correlation coefficient because this is the metric used by the wider NLP community to evaluate performance on CoLA. With this metric, +1 is the best score, and -1 is the worst score. This way, we can see how well we perform against the state of the art models for this specific task.

In [ ]:
# Initiate dataloader for test data
test_dataloader = DataLoader(
            test_dataset, 
            sampler = SequentialSampler(test_dataset), # Pull out batches sequentially.
            batch_size = batch_size # Evaluate with this batch size.
        )

In [ ]:
# Prediction on test set

# Put model in evaluation mode
model.eval()

# Tracking variables 
predictions , true_labels = [], []

# Predict 
for batch in test_dataloader:
  # Add batch to GPU
  batch = tuple(t.to(device) for t in batch)
  
  # Unpack the inputs from our dataloader
  b_input_ids, b_input_mask, b_labels = batch
  
  # Telling the model not to compute or store gradients, saving memory and 
  # speeding up prediction
  with torch.no_grad():
      # Forward pass, calculate logit predictions.
      result = model(b_input_ids, 
                     token_type_ids=None, 
                     attention_mask=b_input_mask,
                     return_dict=True)

  logits = result.logits

  # Move logits and labels to CPU
  logits = logits.detach().cpu().numpy()
  label_ids = b_labels.to('cpu').numpy()
  
  # Store predictions and true labels
  predictions.append(logits)
  true_labels.append(label_ids)

print('    DONE.')

### MCC is used as metric (is also used for imbalanced datasets)

In [ ]:
from sklearn.metrics import matthews_corrcoef

matthews_set = []

# Evaluate each test batch using Matthew's correlation coefficient
print('Calculating Matthews Corr. Coef. for each batch...')

# For each input batch...
for i in range(len(true_labels)):
  
  # The predictions for this batch are a 2-column ndarray (one column for "0" 
  # and one column for "1"). Pick the label with the highest value and turn this
  # in to a list of 0s and 1s.
  pred_labels_i = np.argmax(predictions[i], axis=1).flatten()
  
  # Calculate and store the coef for this batch.  
  matthews = matthews_corrcoef(true_labels[i], pred_labels_i)                
  matthews_set.append(matthews)

In [ ]:
# Create a barplot showing the MCC score for each batch of test samples.
import seaborn as sns 

ax = sns.barplot(x=list(range(len(matthews_set))), y=matthews_set, ci=None)

plt.title('MCC Score per Batch')
plt.ylabel('MCC Score (-1 to +1)')
plt.xlabel('Batch #')

plt.show()

### Combine the results for all of the batches and calculate our final MCC score.

In [ ]:
# Combine the results across all batches. 
flat_predictions = np.concatenate(predictions, axis=0)

# For each sample, pick the label (0 or 1) with the higher score.
flat_predictions = np.argmax(flat_predictions, axis=1).flatten()

# Combine the correct labels for each batch into a single list.
flat_true_labels = np.concatenate(true_labels, axis=0)

# Calculate the MCC
mcc = matthews_corrcoef(flat_true_labels, flat_predictions)

print('Total MCC: %.3f' % mcc)

In [ ]:
# Additionally calculate balanced accuracy
from sklearn.metrics import balanced_accuracy_score, accuracy_score, recall_score, precision_score, f1_score
balanced_accuracy_score(flat_true_labels, flat_predictions)

In [ ]:
# Plot ROC-curve
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

predictions_1 = [pred[1] for pred in np.concatenate(predictions, axis=0)]

fpr, tpr, thresholds = roc_curve(flat_true_labels, predictions_1)
roc_auc = roc_auc_score(flat_true_labels, predictions_1)

plt.figure(figsize=(8,8))
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.gca().set_aspect('equal', adjustable='box') # This line makes the plot square
plt.show()

In [ ]:
print('Sensitivity: ', recall_score(flat_true_labels, flat_predictions))
print('Specificity: ', recall_score(flat_true_labels, flat_predictions, pos_label=0))
print('Accuracy: ', accuracy_score(flat_true_labels, flat_predictions))
print('F1 score: ', f1_score(flat_true_labels, flat_predictions))

#### Use thresholding to set different sensitivity levels

In [ ]:
# 1. Find the threshold corresponding to a sensitivity of 95%.
# Note: Since the thresholds are usually sorted in decreasing order, you might find the closest value to 95% sensitivity starting from the end.

target_sensitivity = 0.95
idx = np.argmin(np.abs(tpr - target_sensitivity))

selected_threshold = thresholds[idx]

# 2. Use this threshold to make binary predictions.
binary_predictions = [1 if p >= selected_threshold else 0 for p in predictions_1]

# 3. Calculate specificity and accuracy using the binary predictions.

# Specificity = True Negative Rate = TN / (TN + FP)
# True Positives (TP) and False Positives (FP) are obtained from the binary predictions
TN = sum([(p == 0 and l == 0) for p, l in zip(binary_predictions, flat_true_labels)])
FP = sum([(p == 1 and l == 0) for p, l in zip(binary_predictions, flat_true_labels)])
specificity = TN / (TN + FP)

accuracy = accuracy_score(flat_true_labels, binary_predictions)

print(f"Specificity at 95% Sensitivity: {specificity:.4f}")
print(f"Accuracy at 95% Sensitivity: {accuracy:.4f}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Determine the corresponding FPR values for the desired sensitivity levels
target_sensitivities = [0.95, 0.93, 0.90, 0.85]
corresponding_fpr = []

for ts in target_sensitivities:
    # Find indices where tpr is close to the target sensitivity
    close_indices = np.where(np.isclose(tpr, ts, atol=0.025))[0]
    
    # Check if close_indices is not empty
    if close_indices.size > 0:
        # Among those indices, choose the one that is closest to the target sensitivity
        best_idx = close_indices[np.argmin(np.abs(tpr[close_indices] - ts))]
        corresponding_fpr.append(fpr[best_idx])

# Now, plot the ROC curve along with these points
plt.figure(figsize=(8, 8))
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')

# Plot the points for the target sensitivities
for ts, cfpr in zip(target_sensitivities, corresponding_fpr):
    plt.plot(cfpr, ts, 'o', markersize=10, label=f'Sens {ts*100:.0f}% (Spec {((1-cfpr)*100):.2f}%)')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.gca().set_aspect('equal', adjustable='box')
plt.show()

# Calculate and print accuracy for target sensitivities
for ts, cfpr in zip(target_sensitivities, corresponding_fpr):
    TP = ts * np.sum(flat_true_labels == 1)
    FP = cfpr * np.sum(flat_true_labels == 0)
    TN = (1 - cfpr) * np.sum(flat_true_labels == 0)
    FN = (1 - ts) * np.sum(flat_true_labels == 1)
    
    accuracy = (TP + TN) / (TP + FP + TN + FN)
    print(f'Accuracy for Sensitivity {ts*100:.0f}%: {accuracy:.2f}')

# Calculate and print balanced accuracy for target sensitivities
for ts, cfpr in zip(target_sensitivities, corresponding_fpr):
    specificity = 1 - cfpr
    balanced_acc = (ts + specificity) / 2
    print(f'Balanced Accuracy for Sensitivity {ts*100:.0f}%: {balanced_acc:.2f}')

# Calculate and print F1 score for target sensitivities
for ts, cfpr in zip(target_sensitivities, corresponding_fpr):
    specificity = 1 - cfpr
    TP = ts * np.sum(flat_true_labels == 1)
    FP = cfpr * np.sum(flat_true_labels == 0)
    precision = TP / (TP + FP)
    f1_score = 2 * (precision * ts) / (precision + ts)
    print(f'F1 Score for Sensitivity {ts*100:.0f}%: {f1_score:.2f}')




In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Function to convert logit to probability
def logit_to_prob(logit):
    return 1 / (1 + np.exp(-logit))

# Determine the corresponding FPR values for the desired sensitivity levels
target_sensitivities = [0.99, 0.98, 0.97, 0.96, 0.95,0.94,0.93,0.92,0.91,0.90,0.89,0.88,0.87,0.86,0.85, 0.84, 0.83, 0.82, 0.81, 0.80]
corresponding_fpr = []
cutoffs_for_target_sens = []

for ts in target_sensitivities:
    # Find the index where the absolute difference between tpr and the target sensitivity is minimum
    best_idx = np.argmin(np.abs(tpr - ts))
    
    corresponding_fpr.append(fpr[best_idx])
    cutoffs_for_target_sens.append(thresholds[best_idx])

# Now, plot the ROC curve along with these points
plt.figure(figsize=(12, 12))
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')

# Plot the points for the target sensitivities
for ts, cfpr in zip(target_sensitivities, corresponding_fpr):
    plt.plot(cfpr, ts, 'o', markersize=10, label=f'Sens {ts*100:.0f}% (Spec {((1-cfpr)*100):.2f}%)')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.gca().set_aspect('equal', adjustable='box')
plt.show()

# Print the cutoffs for each target sensitivity and their probabilities
for ts, cutoff in zip(target_sensitivities, cutoffs_for_target_sens):
    prob_cutoff = logit_to_prob(cutoff)
    print(f'Cutoff for Sensitivity {ts*100:.0f}%: Logit = {cutoff:.2f}, Probability = {prob_cutoff:.2f}')

# Calculate and print accuracy for target sensitivities
for ts, cfpr in zip(target_sensitivities, corresponding_fpr):
    TP = ts * np.sum(flat_true_labels == 1)
    FP = cfpr * np.sum(flat_true_labels == 0)
    TN = (1 - cfpr) * np.sum(flat_true_labels == 0)
    FN = (1 - ts) * np.sum(flat_true_labels == 1)
    
    accuracy = (TP + TN) / (TP + FP + TN + FN)
    print(f'Accuracy for Sensitivity {ts*100:.0f}%: {accuracy:.2f}')

# Calculate and print balanced accuracy for target sensitivities
for ts, cfpr in zip(target_sensitivities, corresponding_fpr):
    specificity = 1 - cfpr
    balanced_acc = (ts + specificity) / 2
    print(f'Balanced Accuracy for Sensitivity {ts*100:.0f}%: {balanced_acc:.2f}')

# Calculate and print F1 score for target sensitivities
for ts, cfpr in zip(target_sensitivities, corresponding_fpr):
    specificity = 1 - cfpr
    TP = ts * np.sum(flat_true_labels == 1)
    FP = cfpr * np.sum(flat_true_labels == 0)
    precision = TP / (TP + FP)
    f1_score = 2 * (precision * ts) / (precision + ts)
    print(f'F1 Score for Sensitivity {ts*100:.0f}%: {f1_score:.2f}')


## Saving Model

In [ ]:
# # Saving best-practices: if you use defaults names for the model, you can reload it using from_pretrained()
# output_dir = './model_CI_addd_save_batch_16_1epochs/'

# # Create output directory if needed
# if not os.path.exists(output_dir):
#     os.makedirs(output_dir)

# print("Saving model to %s" % output_dir)

# # Save a trained model, configuration and tokenizer using `save_pretrained()`.
# # They can then be reloaded using `from_pretrained()`
# model_to_save = model.module if hasattr(model, 'module') else model  # Take care of distributed/parallel training
# model_to_save.save_pretrained(output_dir)
# tokenizer.save_pretrained(output_dir)

# # # Good practice: save your training arguments together with the trained model
# # torch.save(args, os.path.join(output_dir, 'training_args.bin'))
